In [1]:
import os

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

import sys

sclembas = '/home/hmbaghda/Projects/scLEMBAS'
sys.path.insert(1, os.path.join(sclembas))
from scLEMBAS import parse_network, io

/nobackup/users/hmbaghda/Software/miniforge3/envs/cart_lembas/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
n_cores = 12
os.environ["OMP_NUM_THREADS"] = str(n_cores)
os.environ["MKL_NUM_THREADS"] = str(n_cores)
os.environ["OPENBLAS_NUM_THREADS"] = str(n_cores)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(n_cores)
os.environ["NUMEXPR_NUM_THREADS"] = str(n_cores)

seed = 888
data_path = '/nobackup/users/hmbaghda/scLEMBAS/analysis'

In [11]:
tf_adata = io.read_tfad(file_name = os.path.join(data_path, 'interim', 'ID_tf_activity.h5ad'))

Load and parse input signaling network:

In [12]:
source_label = 'source_genesymbol'
target_label = 'target_genesymbol'
weight_label = 'mode_of_action'
stimulation_label = 'consensus_stimulation'
inhibition_label = 'consensus_inhibition'

In [5]:
sn_ppis = parse_network.load_network('omnipath', organism = 'mouse', static = True)

sn_ppis = parse_network.correct_network(sn_ppis = sn_ppis,
                                        source_label = source_label, target_label = target_label,
                                        stimulation_label = stimulation_label, inhibition_label = inhibition_label)

sn_ppis = parse_network.extract_network(sn_ppis, curation_effort_thresh = 5, n_references_thresh = 3,
                                        resources = ['HuRI','IntAct','KEGG-MEDICUS','NetPath','Reactome_SignaLink3','SPIKE','SignaLink3','SIGNOR', 
                                                'Baccin2019', 'Ramilowski2015', 'Reactome_LRdb', 'UniProt_LRdb', 'CellChatDB', 'CellPhoneDB', 'connectomeDB2020', 'scConnect'], 
                                        source_label = source_label, target_label = target_label,
                                        drop_self = True, verbose = True)


The thresholds filtered 66381  of 75185 interactions
The resources filtered 1940  of 8804 interactions


In [6]:
tf_labels = tf_adata.var.index.unique().tolist()

ligand_labels = tf_adata.obs['sample'].unique().tolist()
ligand_labels = [(l[0] + l[1:].lower()).replace('-', '') for l in ligand_labels] # mouse naming convention

In [7]:
# # filter for paths b/w ligand and tf
# fn_1, _ = parse_network.create_connected_network(sn_ppis, ligand_labels, tf_labels, source_label = source_label, target_label = target_label, 
#                        path_finder = 'shortest')
# fn_2, _ = parse_network.create_connected_network(sn_ppis, ligand_labels, tf_labels, source_label = source_label, target_label = target_label, 
#                        path_finder = 'connected')
# # of the methods to identify paths, retain the one that has the most interactions
# if fn_1.shape[0] > fn_2.shape[0]:
#     sn_ppis = fn_1
# else:
#     sn_ppis = fn_2

# del fn_1, fn_2

sn_ppis, _ = parse_network.create_connected_network(sn_ppis, ligand_labels, tf_labels, source_label = source_label, target_label = target_label, 
                       path_finder = 'connected')

Format network as needed for model building:

In [8]:
sn_ppis = parse_network.format_network(sn_ppis, weight_label, stimulation_label, inhibition_label) 

In [9]:
print('The signaling network contains {} interactions'.format(sn_ppis.shape[0]))
sn_ppis[[source_label, target_label, weight_label, stimulation_label, inhibition_label]].head()

The signaling network contains 4583 interactions


,source_genesymbol,target_genesymbol,mode_of_action,consensus_stimulation,consensus_inhibition
145,Mapk14,Mapkapk2,1.0,True,False
146,Mapkapk2,Mapk14,0.1,False,False
152,Epor,Jak2,0.1,False,False
153,Jak2,Epor,1.0,True,False
159,Numb,Notch1,-1.0,False,True


In [10]:
sn_ppis.to_csv(os.path.join(data_path, 'processed', 'sn_ppis.csv'))